# KV Cache — Interactive Notebook

This notebook walks through KV Cache and related techniques for efficient LLM inference:

| Section | Topic |
|---|---|
| 1 | What is KV Cache? |
| 2 | Naive vs cached autoregressive generation |
| 3 | Memory cost calculator |
| 4 | Attention variants: MHA → MQA → GQA |
| 5 | Multi-head Latent Attention (MLA / DeepSeek) |
| 6 | Sliding Window & Streaming LLM |
| 7 | KV Cache pruning (H2O / Scissorhands) |
| 8 | Cross-conversation / Prompt caching |
| 9 | Student exercises |

In [ ]:
!pip install -q matplotlib numpy ipywidgets

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
import time, math

---
## Part 1 — What Is KV Cache?

### Background: Causal (Decoder-Only) Self-Attention

In a decoder-only Transformer (GPT, LLaMA, Gemma …), each token $i$ produces three vectors:
$$\mathbf{q}_i = W_q\mathbf{x}_i, \quad \mathbf{k}_i = W_k\mathbf{x}_i, \quad \mathbf{v}_i = W_v\mathbf{x}_i$$

The output for token $t$ is:
$$\mathbf{o}_t = \sum_{i=1}^{t} \alpha_{ti}\,\mathbf{v}_i, \qquad \alpha_{ti} = \text{softmax}\!\left(\frac{\mathbf{q}_t^\top \mathbf{k}_i}{\sqrt{d}}\right)$$

**The problem during generation:**  
When generating token $t+1$, we need $\mathbf{k}_i$ and $\mathbf{v}_i$ for **all** previous tokens $i = 1 \ldots t$.  
Without caching, we recompute them from scratch every step → $O(T^2)$ total work.

**The fix — KV Cache:**  
Store $\{\mathbf{k}_i, \mathbf{v}_i\}_{i<t}$ in GPU memory. Each new step only computes one new $(\mathbf{k}_t, \mathbf{v}_t)$ pair and appends it.  
Computation drops to $O(T)$ — trading **time** for **memory**.

---
## Part 2 — Naive vs KV-Cached Generation

We simulate a single attention head with `d_model` dimensions generating `T` tokens.

In [ ]:
def softmax(x):
    e = np.exp(x - x.max())
    return e / e.sum()

# ── Fixed weight matrices (random but seeded for reproducibility) ──
rng = np.random.default_rng(42)
D   = 16   # head dimension
Wq  = rng.standard_normal((D, D)) * 0.1
Wk  = rng.standard_normal((D, D)) * 0.1
Wv  = rng.standard_normal((D, D)) * 0.1

def make_token_embeddings(T, D, seed=0):
    return np.random.default_rng(seed).standard_normal((T, D))

# ── APPROACH A: Naive — recompute all K,V at every step ──
def naive_generate(X):
    """
    X: (T, D) token embeddings
    Returns output vectors and total matmul count (proxy for FLOPs).
    """
    T = len(X)
    outputs = []
    matmuls = 0

    for t in range(T):
        # Re-project ALL tokens 1..t every step
        K = X[:t+1] @ Wk.T   # shape (t+1, D)
        V = X[:t+1] @ Wv.T
        Q_t = X[t] @ Wq.T    # shape (D,)

        scores  = K @ Q_t / math.sqrt(D)   # (t+1,)
        weights = softmax(scores)           # (t+1,)
        out_t   = weights @ V              # (D,)

        outputs.append(out_t)
        matmuls += (t + 1)   # proportional to context length

    return np.stack(outputs), matmuls


# ── APPROACH B: KV Cache — store past K,V, only compute new one ──
def kvcache_generate(X):
    """
    X: (T, D) token embeddings
    Returns output vectors and total matmul count.
    """
    T = len(X)
    outputs = []
    matmuls = 0

    # The cache grows by one entry per step
    K_cache = []   # list of (D,) key vectors
    V_cache = []   # list of (D,) value vectors

    for t in range(T):
        # Only project the NEW token
        k_t = X[t] @ Wk.T
        v_t = X[t] @ Wv.T
        q_t = X[t] @ Wq.T

        K_cache.append(k_t)
        V_cache.append(v_t)

        K = np.stack(K_cache)   # (t+1, D)
        V = np.stack(V_cache)

        scores  = K @ q_t / math.sqrt(D)
        weights = softmax(scores)
        out_t   = weights @ V

        outputs.append(out_t)
        matmuls += 1   # only ONE new K,V projection per step

    return np.stack(outputs), matmuls


# ── Verify both produce the same results ──
T = 8
X = make_token_embeddings(T, D)

out_naive,  muls_naive  = naive_generate(X)
out_cached, muls_cached = kvcache_generate(X)

print(f"Outputs match: {np.allclose(out_naive, out_cached, atol=1e-10)}")
print(f"Naive  total K/V projections ∝ {muls_naive}   (∝ T²/2 = {T*(T+1)//2})")
print(f"Cached total K/V projections ∝ {muls_cached}  (∝ T    = {T})")
print(f"Speedup on K/V computation:  {muls_naive / muls_cached:.1f}×")

In [ ]:
# ── Show how the speedup scales with sequence length ──
lengths = list(range(1, 201))
naive_work  = [T * (T + 1) // 2 for T in lengths]
cached_work = lengths   # always T

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(lengths, naive_work,  label='Naive  O(T²)', color='tomato')
ax1.plot(lengths, cached_work, label='KV Cache O(T)', color='steelblue')
ax1.set_xlabel('Sequence Length T')
ax1.set_ylabel('K/V projections (proportional to FLOPs)')
ax1.set_title('Total K/V Computation Cost')
ax1.legend()
ax1.grid(True, alpha=0.3)

speedups = [n / c for n, c in zip(naive_work, cached_work)]
ax2.plot(lengths, speedups, color='seagreen')
ax2.set_xlabel('Sequence Length T')
ax2.set_ylabel('Speedup (Naive / Cached)')
ax2.set_title('KV Cache Speedup vs Sequence Length')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("At T=200 the naive approach does", naive_work[-1], "K/V projections;")
print("KV cache does only", cached_work[-1], f"— a {speedups[-1]:.0f}× reduction.")

---
## Part 3 — Memory Cost Calculator

**Per-token KV cache size formula:**
$$\text{bytes} = L_{\text{layers}} \times H_{\text{heads}} \times D_{\text{head}} \times B_{\text{dtype}} \times 2_{\text{(K+V)}}$$

**Gemma 2 27B example** (from the lecture):  
$46 \times 32 \times 128 \times 2 \times 2 = 753{,}664$ bytes ≈ **0.72 MB per token**  
On an A100 (80 GB) → only about **114 k tokens** fit in cache.

In [ ]:
# ── Interactive memory calculator ──
style = {'description_width': '160px'}
layout = widgets.Layout(width='420px')

w_layers  = widgets.IntSlider(value=46,   min=1,  max=96,   description='Layers',          style=style, layout=layout)
w_heads   = widgets.IntSlider(value=32,   min=1,  max=128,  description='KV Heads',         style=style, layout=layout)
w_dim     = widgets.IntSlider(value=128,  min=16, max=256,  description='Head Dim',         style=style, layout=layout)
w_dtype   = widgets.Dropdown(options=[('FP16 (2 bytes)', 2), ('FP32 (4 bytes)', 4),
                                       ('INT8 (1 byte)', 1), ('FP8 (1 byte)', 1)],
                              value=2, description='Dtype',   style=style, layout=layout)
w_gpu_gb  = widgets.FloatSlider(value=80.0, min=8, max=640, step=8,
                                 description='GPU Memory (GB)',   style=style, layout=layout)
w_model_gb= widgets.FloatSlider(value=54.0, min=0, max=320, step=1,
                                 description='Model Weights (GB)', style=style, layout=layout)

out_calc = widgets.Output()

def update_calc(*_):
    layers = w_layers.value
    heads  = w_heads.value
    dim    = w_dim.value
    dtype  = w_dtype.value
    gpu_gb = w_gpu_gb.value
    mdl_gb = w_model_gb.value

    bytes_per_token = layers * heads * dim * dtype * 2   # 2 = K + V
    kb_per_token    = bytes_per_token / 1024
    mb_per_token    = kb_per_token / 1024

    free_gb      = max(0, gpu_gb - mdl_gb)
    free_bytes   = free_gb * 1024**3
    max_tokens   = int(free_bytes / bytes_per_token) if bytes_per_token > 0 else 0

    with out_calc:
        clear_output(wait=True)
        print(f"Per-token KV cache size:")
        print(f"  {layers} layers × {heads} KV heads × {dim} dim × {dtype} bytes × 2 (K+V)")
        print(f"  = {bytes_per_token:,} bytes  ({kb_per_token:.1f} KB  /  {mb_per_token:.3f} MB)")
        print()
        print(f"GPU: {gpu_gb} GB total  —  {mdl_gb} GB model weights  =  {free_gb:.1f} GB free")
        print(f"Max tokens in KV cache:  ~{max_tokens:,} tokens")

        # Bar chart: model vs max cache usage
        fig, ax = plt.subplots(figsize=(7, 2.5))
        bar_vals   = [mdl_gb, free_gb]
        bar_labels = ['Model weights', 'KV Cache budget']
        bar_colors = ['#e07b54', '#5b9bd5']
        ax.barh(bar_labels, bar_vals, color=bar_colors, height=0.5)
        ax.set_xlabel('GPU Memory (GB)')
        ax.set_title(f'A{int(gpu_gb)}GB allocation  |  max {max_tokens:,} cached tokens')
        ax.set_xlim(0, gpu_gb)
        ax.grid(True, axis='x', alpha=0.3)
        plt.tight_layout()
        plt.show()

for w in [w_layers, w_heads, w_dim, w_dtype, w_gpu_gb, w_model_gb]:
    w.observe(update_calc, names='value')

display(widgets.VBox([w_layers, w_heads, w_dim, w_dtype, w_gpu_gb, w_model_gb]), out_calc)
update_calc()

---
## Part 4 — Attention Variants & KV Cache Size

### Three flavors

| Variant | Key idea | Used by |
|---|---|---|
| **MHA** Multi-Head Attention | Each query head has its **own** K, V | GPT-2, BERT |
| **MQA** Multi-Query Attention | All query heads share **one** K, V | Falcon, early PaLM |
| **GQA** Grouped-Query Attention | Groups of query heads share K, V | LLaMA 2/3, Gemma |

KV cache size scales directly with the **number of KV heads**, not query heads.

In [ ]:
def attention_variant(X, Wqs, Wks, Wvs, mode='MHA', num_kv_groups=None):
    """
    Simulate one forward pass of MHA / MQA / GQA.
    X    : (T, D) token embeddings
    Wqs  : list of H query projection matrices
    Wks  : list of KV projection matrices (H for MHA, 1 for MQA, G for GQA)
    Wvs  : same as Wks
    Returns: outputs (T, H, d_head), kv_cache_size in elements
    """
    T, _ = X.shape
    H    = len(Wqs)       # number of query heads
    Hkv  = len(Wks)       # number of KV heads
    d    = Wqs[0].shape[0]

    # Compute K, V for each KV head (only Hkv projections)
    K_all = [X @ Wks[h].T for h in range(Hkv)]   # Hkv × (T, d)
    V_all = [X @ Wvs[h].T for h in range(Hkv)]

    head_outputs = []
    for h in range(H):
        Q_h = X @ Wqs[h].T   # (T, d)

        # Which KV head does query head h use?
        if mode == 'MHA':
            kv_idx = h
        elif mode == 'MQA':
            kv_idx = 0
        else:   # GQA
            queries_per_group = H // Hkv
            kv_idx = h // queries_per_group

        K_h = K_all[kv_idx]   # (T, d)
        V_h = V_all[kv_idx]

        # Causal attention mask
        scores = Q_h @ K_h.T / math.sqrt(d)          # (T, T)
        mask   = np.triu(np.full((T, T), -1e9), 1)   # upper-tri = -inf
        scores = scores + mask

        weights = np.exp(scores - scores.max(axis=1, keepdims=True))
        weights = weights / weights.sum(axis=1, keepdims=True)   # (T, T)
        out_h   = weights @ V_h   # (T, d)
        head_outputs.append(out_h)

    kv_cache_elements = T * Hkv * d * 2   # 2 = K + V
    return np.stack(head_outputs, axis=1), kv_cache_elements


# ── Setup: 4 query heads, head dim 16 ──
rng2  = np.random.default_rng(0)
H, D, T_demo = 4, 16, 6
X_demo = rng2.standard_normal((T_demo, D))

Wqs_demo = [rng2.standard_normal((D, D)) * 0.1 for _ in range(H)]

# MHA: H=4 KV heads
Wks_mha = [rng2.standard_normal((D, D)) * 0.1 for _ in range(H)]
Wvs_mha = [rng2.standard_normal((D, D)) * 0.1 for _ in range(H)]

# MQA: 1 KV head
Wks_mqa = Wks_mha[:1]
Wvs_mqa = Wvs_mha[:1]

# GQA: 2 KV heads (2 queries per group)
Wks_gqa = Wks_mha[:2]
Wvs_gqa = Wvs_mha[:2]

_, kv_mha = attention_variant(X_demo, Wqs_demo, Wks_mha, Wvs_mha, mode='MHA')
_, kv_mqa = attention_variant(X_demo, Wqs_demo, Wks_mqa, Wvs_mqa, mode='MQA')
_, kv_gqa = attention_variant(X_demo, Wqs_demo, Wks_gqa, Wvs_gqa, mode='GQA')

print(f"{'Variant':<8} {'KV heads':<12} {'KV cache size (elements)':<28} {'Relative to MHA':<18}")
print('-' * 68)
for name, kv_heads, kv_sz in [('MHA', H, kv_mha), ('GQA', 2, kv_gqa), ('MQA', 1, kv_mqa)]:
    print(f"{name:<8} {kv_heads:<12} {kv_sz:<28,} {kv_sz/kv_mha:.2f}×")

In [ ]:
# ── Interactive: compare KV cache sizes across variants ──
w_H     = widgets.IntSlider(value=32, min=1, max=64, description='Query Heads H',
                             style={'description_width':'140px'}, layout=widgets.Layout(width='400px'))
w_G     = widgets.IntSlider(value=8,  min=1, max=32, description='KV Groups G (GQA)',
                             style={'description_width':'140px'}, layout=widgets.Layout(width='400px'))
out_var = widgets.Output()

def update_variants(*_):
    H_q = w_H.value
    G   = min(w_G.value, H_q)   # G ≤ H

    mha_kv = H_q   # one KV head per query head
    gqa_kv = G
    mqa_kv = 1

    with out_var:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(7, 3.5))
        variants = ['MHA\n(baseline)', f'GQA\n(G={G})', 'MQA\n(G=1)']
        kv_heads = [mha_kv, gqa_kv, mqa_kv]
        colors   = ['#e07b54', '#5b9bd5', '#70ad47']
        bars = ax.bar(variants, kv_heads, color=colors, width=0.5)
        for bar, v in zip(bars, kv_heads):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                    f'{v} KV heads\n({v/mha_kv:.2f}×)', ha='center', fontsize=9)
        ax.set_ylabel('Number of KV Heads')
        ax.set_title(f'KV Cache Size (H={H_q} query heads) — proportional to # KV heads')
        ax.set_ylim(0, H_q * 1.25)
        ax.grid(True, axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()

w_H.observe(update_variants, names='value')
w_G.observe(update_variants, names='value')
display(widgets.HBox([w_H, w_G]), out_var)
update_variants()

---
## Part 5 — Multi-head Latent Attention (MLA)

Used by **DeepSeek**. Instead of caching full $\mathbf{k}_i, \mathbf{v}_i$, compress them into a tiny **latent vector** $\mathbf{c}_i$:

$$\mathbf{c}_i = W_{\text{down}}\,[\mathbf{k}_i ; \mathbf{v}_i] \in \mathbb{R}^{d_c}, \quad d_c \ll 2 H d_{\text{head}}$$

**Key attention trick:** Instead of expanding $\mathbf{c}$ back to $\mathbf{k}$ at query time:
$$\alpha = \mathbf{q}^\top \mathbf{k} = \mathbf{q}^\top W_k \mathbf{c} = (W_k^\top \mathbf{q})^\top \mathbf{c} = \mathbf{q}'^\top \mathbf{c}$$
Pre-transform the query once: $\mathbf{q}' = W_k^\top \mathbf{q}$ — then dot directly with **cached** $\mathbf{c}$.

Similarly for values:
$$\mathbf{o} = \sum_i \alpha_i \mathbf{v}_i = \sum_i \alpha_i W_v \mathbf{c}_i = W_v\!\left(\sum_i \alpha_i \mathbf{c}_i\right)$$
Compute the weighted sum over small $\mathbf{c}_i$, apply $W_v$ **once** at the end.

In [ ]:
def mla_attention(X, Wq, Wk, Wv, W_down, W_up_k, W_up_v):
    """
    MLA: compress K,V into latent c, cache c, reconstruct at query time.
    X      : (T, D)
    W_down : (d_c, D)  — compress each token to latent
    W_up_k : (D, d_c)  — reconstruct k from c   (= Wk @ inv(W_down) conceptually)
    W_up_v : (D, d_c)  — reconstruct v from c
    Returns: outputs (T, D), number of elements cached
    """
    T, D = X.shape
    d_c  = W_down.shape[0]

    # Build latent cache — only d_c floats per token (tiny!)
    C = X @ W_down.T   # (T, d_c)  ← this is what we cache

    Q  = X @ Wq.T   # (T, D)

    # ---- Trick: transform Q instead of expanding C ----
    # Standard: k = c @ W_up_k.T;  score = q @ k.T
    # MLA:      q' = q @ W_up_k;   score = q' @ c.T   (same result!)
    Q_prime = Q @ W_up_k   # (T, d_c)  — absorb W_up_k into query

    scores  = Q_prime @ C.T / math.sqrt(d_c)   # (T, T)
    mask    = np.triu(np.full((T, T), -1e9), 1)
    scores  += mask
    weights  = np.exp(scores - scores.max(1, keepdims=True))
    weights /= weights.sum(1, keepdims=True)   # (T, T)

    # ---- Value trick: weighted sum of c, then one W_up_v ----
    # Standard: v_i = c_i @ W_up_v.T;  o = sum_i alpha_i * v_i
    # MLA:      o = W_up_v @ (sum_i alpha_i * c_i).T   (same result!)
    C_weighted = weights @ C   # (T, d_c)
    output     = C_weighted @ W_up_v.T   # (T, D)

    kv_elements_cached = T * d_c   # only latents, not full K+V
    return output, kv_elements_cached


# ── Compare cache sizes ──
rng3   = np.random.default_rng(7)
D_mla, d_c, T_mla = 64, 8, 10   # compress 64-dim K+V down to 8-dim latent

Wq_mla   = rng3.standard_normal((D_mla, D_mla)) * 0.1
Wk_mla   = rng3.standard_normal((D_mla, D_mla)) * 0.1
Wv_mla   = rng3.standard_normal((D_mla, D_mla)) * 0.1
W_down   = rng3.standard_normal((d_c,   D_mla)) * 0.1
W_up_k   = rng3.standard_normal((D_mla, d_c))   * 0.1
W_up_v   = rng3.standard_normal((D_mla, d_c))   * 0.1
X_mla    = rng3.standard_normal((T_mla, D_mla))

_, kv_mha_elems = T_mla * D_mla * 2, None   # full K+V per token
kv_mha_elems    = T_mla * D_mla * 2

_, kv_mla_elems = mla_attention(X_mla, Wq_mla, Wk_mla, Wv_mla, W_down, W_up_k, W_up_v)

print(f"Full MHA KV cache:  {kv_mha_elems} elements  (T={T_mla}, D={D_mla}, K+V)")
print(f"MLA latent cache:   {kv_mla_elems} elements  (T={T_mla}, d_c={d_c})")
print(f"Cache compression:  {kv_mha_elems / kv_mla_elems:.1f}×  (d_c / (2*D) = {d_c}/{2*D_mla})")

---
## Part 6 — Sliding Window Attention & Streaming LLM

**Sliding Window Attention (Mistral 7B):**  
Each token only attends to the **W most recent** tokens. KV cache is bounded at size $W$ regardless of total sequence length.

**Streaming LLM:**  
Keep a few **"sink" tokens** (position 0 acts as an attention sink) plus a recent window. Enables indefinite generation with constant cache size.

In [ ]:
def make_attention_mask(T, mode='full', window=4, n_sink=1):
    """Return (T, T) boolean mask — True means this position is attended to."""
    mask = np.zeros((T, T), dtype=bool)
    for t in range(T):
        if mode == 'full':
            mask[t, :t+1] = True
        elif mode == 'window':
            start = max(0, t - window + 1)
            mask[t, start:t+1] = True
        elif mode == 'streaming':
            # sink tokens (positions 0..n_sink-1) + recent window
            mask[t, :n_sink] = True
            start = max(n_sink, t - window + 1)
            mask[t, start:t+1] = True
    return mask


T_vis = 20
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
titles  = ['Full Attention\n(standard KV cache)', 'Sliding Window (W=4)\n(Mistral 7B)', 'Streaming LLM\n(1 sink + W=3 recent)']
configs = [
    ('full',      dict()),
    ('window',    dict(window=4)),
    ('streaming', dict(window=3, n_sink=1)),
]

for ax, title, (mode, kw) in zip(axes, titles, configs):
    mask = make_attention_mask(T_vis, mode=mode, **kw)
    ax.imshow(mask, cmap='Blues', aspect='auto', interpolation='nearest')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Key position')
    ax.set_ylabel('Query position')

    # Cache size at last step
    cache_sz = mask[-1].sum()
    ax.text(T_vis//2, T_vis - 1.8, f'Cache at t={T_vis}: {cache_sz} tokens',
            ha='center', color='white', fontweight='bold', fontsize=9)

plt.suptitle('Attention Mask Patterns (blue = attended)', fontsize=12)
plt.tight_layout()
plt.show()

print("Cache size at the last token (T=20):")
for title, (mode, kw) in zip(titles, configs):
    m = make_attention_mask(T_vis, mode=mode, **kw)
    print(f"  {mode:<12}: {m[-1].sum():>3} tokens cached")

In [ ]:
# ── Show how cache size grows with sequence length ──
lengths = range(1, 101)
W, S = 8, 2   # window size, num sinks

full_cache     = [t for t in lengths]
window_cache   = [min(t, W) for t in lengths]
streaming_cache= [min(S + max(0, t - S), S + W) for t in lengths]

plt.figure(figsize=(8, 4))
plt.plot(lengths, full_cache,      label='Full Attention (O(T))', color='tomato')
plt.plot(lengths, window_cache,    label=f'Sliding Window (W={W})', color='steelblue')
plt.plot(lengths, streaming_cache, label=f'Streaming LLM ({S} sinks + W={W})', color='seagreen')
plt.xlabel('Sequence Length')
plt.ylabel('KV Cache Size (tokens)')
plt.title('Cache Size Growth by Attention Strategy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 7 — KV Cache Pruning

**Observation (Scissorhands / H2O papers):**  
At each step, only a **small fraction** of past tokens actually receive high attention weight. A few tokens repeatedly dominate — so we can **evict** low-attention KV entries to save memory.

**Strategy:** Keep only the top-K tokens by accumulated attention score.

In [ ]:
rng4 = np.random.default_rng(99)
T_prune = 40
D_prune = 32

X_prune = rng4.standard_normal((T_prune, D_prune))
Wq_p    = rng4.standard_normal((D_prune, D_prune)) * 0.1
Wk_p    = rng4.standard_normal((D_prune, D_prune)) * 0.1

# Compute a full causal attention map
Q_all = X_prune @ Wq_p.T
K_all = X_prune @ Wk_p.T
scores_full = Q_all @ K_all.T / math.sqrt(D_prune)
mask_full   = np.triu(np.full((T_prune, T_prune), -1e9), 1)
scores_full += mask_full
weights_full = np.exp(scores_full - scores_full.max(1, keepdims=True))
weights_full /= weights_full.sum(1, keepdims=True)

# Accumulated attention score: sum over all query positions
accumulated = weights_full.sum(axis=0)   # how much total attention each token received

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

im = ax1.imshow(weights_full, cmap='Greens', aspect='auto')
ax1.set_title('Attention weights (causal)')
ax1.set_xlabel('Key position (token being attended to)')
ax1.set_ylabel('Query position')
plt.colorbar(im, ax=ax1)

ax2.bar(range(T_prune), accumulated, color='steelblue', alpha=0.8)
ax2.set_xlabel('Token position')
ax2.set_ylabel('Accumulated attention received')
ax2.set_title('Accumulated Attention per Token\n(candidates for KV pruning: low bars → evict)')
# Mark top-8 in red as "keep"
keep_k  = 8
top_idx = np.argsort(accumulated)[-keep_k:]
for idx in top_idx:
    ax2.bar(idx, accumulated[idx], color='tomato', alpha=0.9)
ax2.legend([mpatches.Patch(color='steelblue'), mpatches.Patch(color='tomato')],
           ['Evicted (low attention)', f'Kept (top-{keep_k})'])

plt.tight_layout()
plt.show()

print(f"Keep top-{keep_k} of {T_prune} tokens → cache compression: {T_prune/keep_k:.1f}×")

---
## Part 8 — Cross-Conversation / Prompt Caching

**Idea:** When many requests share the same **prefix** (system prompt, tool definitions, long document), the KV cache for that prefix can be reused across conversations — no recomputation.

**Design rule:** Put **stable content first**, dynamic content last — maximise the shared prefix.

**Business impact** (from lecture slide): ~50–80% cost reduction, 13–31% TTFT (Time-to-First-Token) reduction.

In [ ]:
def prefix_overlap(prompt_a, prompt_b):
    """Return the length of the longest shared prefix (token-level)."""
    min_len = min(len(prompt_a), len(prompt_b))
    for i in range(min_len):
        if prompt_a[i] != prompt_b[i]:
            return i
    return min_len


def simulate_cache_hit(requests, cached_prefix_len):
    """
    Simulate a batch of API requests.
    Returns (tokens_computed, tokens_from_cache) per request.
    """
    results = []
    for req in requests:
        shared = min(cached_prefix_len, len(req))
        cache_hit   = shared
        recomputed  = len(req) - shared
        results.append((recomputed, cache_hit))
    return results


# ── Two prompt designs: bad (variable prefix) vs good (stable prefix) ──
system_prompt_tokens = 800   # fixed system prompt length
tool_def_tokens      = 400   # tool definitions
user_query_tokens    = 100   # varies per request

# Bad design: user-specific info is at the front
bad_design_req1 = ['user_query'] * user_query_tokens + ['system'] * system_prompt_tokens + ['tools'] * tool_def_tokens
bad_design_req2 = ['user_query2'] * user_query_tokens + ['system'] * system_prompt_tokens + ['tools'] * tool_def_tokens
bad_overlap = prefix_overlap(bad_design_req1, bad_design_req2)

# Good design: stable content first, query at the end
good_design_req1 = ['system'] * system_prompt_tokens + ['tools'] * tool_def_tokens + ['user_query'] * user_query_tokens
good_design_req2 = ['system'] * system_prompt_tokens + ['tools'] * tool_def_tokens + ['user_query2'] * user_query_tokens
good_overlap = prefix_overlap(good_design_req1, good_design_req2)

total_len = len(bad_design_req1)

fig, axes = plt.subplots(2, 1, figsize=(10, 5))
for ax, title, overlap, req in [
    (axes[0], 'Bad design: user query FIRST (small shared prefix)', bad_overlap, bad_design_req1),
    (axes[1], 'Good design: system prompt FIRST (large shared prefix)', good_overlap, good_design_req1),
]:
    hit_pct  = overlap / total_len * 100
    miss_pct = 100 - hit_pct
    ax.barh(['Request'], [overlap],       color='seagreen', label=f'Cache HIT ({overlap} tokens, {hit_pct:.0f}%)')
    ax.barh(['Request'], [total_len - overlap], left=overlap, color='tomato', label=f'Recomputed ({total_len-overlap} tokens)')
    ax.set_xlim(0, total_len)
    ax.set_xlabel('Token position')
    ax.set_title(title)
    ax.legend(loc='center right', fontsize=8)
    ax.axvline(overlap, color='black', linestyle='--', linewidth=1.5)

plt.suptitle('Prompt Caching: Effect of Token Ordering', fontsize=12)
plt.tight_layout()
plt.show()

print(f"\nBad design:  {bad_overlap} tokens shared prefix ({bad_overlap/total_len*100:.1f}% cache hit)")
print(f"Good design: {good_overlap} tokens shared prefix ({good_overlap/total_len*100:.1f}% cache hit)")

In [ ]:
# ── Interactive: estimate cost savings from prompt caching ──
w_sys   = widgets.IntSlider(value=800, min=0, max=4000, step=50,
                             description='System prompt (tokens)',
                             style={'description_width':'200px'}, layout=widgets.Layout(width='500px'))
w_user  = widgets.IntSlider(value=200, min=1, max=2000, step=50,
                             description='User query (tokens)',
                             style={'description_width':'200px'}, layout=widgets.Layout(width='500px'))
w_price = widgets.FloatText(value=2.50, description='Price per 1M tokens ($)',
                             style={'description_width':'200px'}, layout=widgets.Layout(width='300px'))
w_cache_price = widgets.FloatText(value=0.25, description='Cached price per 1M tokens ($)',
                                   style={'description_width':'200px'}, layout=widgets.Layout(width='300px'))
w_reqs  = widgets.IntSlider(value=100, min=1, max=1000, step=10,
                             description='Requests per day',
                             style={'description_width':'200px'}, layout=widgets.Layout(width='500px'))
out_cost = widgets.Output()

def update_cost(*_):
    sys_tok    = w_sys.value
    user_tok   = w_user.value
    total_tok  = sys_tok + user_tok
    price_full = w_price.value / 1e6
    price_hit  = w_cache_price.value / 1e6
    n_reqs     = w_reqs.value

    # No caching: pay full price for all tokens on every request
    cost_no_cache = total_tok * price_full * n_reqs

    # With caching: system prompt is a cache hit on 2nd+ request
    cost_first   = total_tok * price_full                         # first request: miss
    cost_rest    = (sys_tok * price_hit + user_tok * price_full)  # subsequent: hit on sys prompt
    cost_cached  = cost_first + cost_rest * (n_reqs - 1) if n_reqs > 1 else cost_first

    saving_pct = (1 - cost_cached / cost_no_cache) * 100 if cost_no_cache > 0 else 0

    with out_cost:
        clear_output(wait=True)
        print(f"  No caching:     ${cost_no_cache:.4f}/day  ({n_reqs} × {total_tok} tokens × ${w_price.value}/1M)")
        print(f"  With caching:   ${cost_cached:.4f}/day  (sys={sys_tok} tokens hit @ ${w_cache_price.value}/1M)")
        print(f"  Savings:        {saving_pct:.1f}%  (${cost_no_cache - cost_cached:.4f}/day)")

        fig, ax = plt.subplots(figsize=(6, 2.5))
        ax.bar(['No cache', 'With cache'], [cost_no_cache, cost_cached],
               color=['tomato', 'steelblue'], width=0.4)
        ax.set_ylabel('Cost per day ($)')
        ax.set_title(f'Daily API cost | {n_reqs} requests | {saving_pct:.1f}% savings')
        ax.grid(True, axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()

for w in [w_sys, w_user, w_price, w_cache_price, w_reqs]:
    w.observe(update_cost, names='value')
display(widgets.VBox([w_sys, w_user, w_price, w_cache_price, w_reqs]), out_cost)
update_cost()

---
## Part 9 — Summary

| Technique | Core idea | Changes attention? | Needs retraining? | Cost |
|---|---|---|---|---|
| **KV Cache** | Store computed K,V; reuse each step | No | No | Memory |
| **MQA** | All query heads share 1 KV head | Yes | Yes | Possible quality loss |
| **GQA** | Groups of queries share KV heads | Yes | Yes | — |
| **MLA** | Compress K,V → tiny latent c | Yes | Yes | Extra matmul |
| **Sliding Window** | Attend only to recent W tokens | Yes | Maybe | Bounded cache |
| **Streaming LLM** | Sink tokens + recent window | Yes | Maybe | Bounded cache |
| **KV Pruning** | Evict low-attention K,V entries | Yes | No | Possible quality loss |
| **Prompt Caching** | Reuse KV across requests with shared prefix | No | No | Memory |

---
## Part 10 — Student Exercises

### Exercise 1 — Memory calculation
A model has 32 layers, 16 KV heads, head dimension 128, stored in FP16.
An H100 GPU has 80 GB, with 40 GB reserved for model weights.

**Q:** How many tokens can fit in the KV cache? Show your calculation.

In [ ]:
# Your calculation here:
layers     = 32
kv_heads   = 16
head_dim   = 128
dtype_bytes = 2      # FP16
kv_factor   = 2      # K + V

bytes_per_token = layers * kv_heads * head_dim * dtype_bytes * kv_factor
print(f"Bytes per token: {bytes_per_token:,}  ({bytes_per_token/1024:.1f} KB)")

free_gb   = 80 - 40
free_bytes = free_gb * 1024**3
max_tokens = free_bytes // bytes_per_token
print(f"Max tokens in KV cache: {max_tokens:,}")

### Exercise 2 — Implement GQA from scratch

Fill in the blank below to implement GQA with `H_q=8` query heads and `G=2` KV groups.

In [ ]:
def gqa_forward(Q_heads, K_groups, V_groups):
    """
    Q_heads  : (H_q, T, d)  — one query matrix per query head
    K_groups : (G,   T, d)  — one key matrix per group
    V_groups : (G,   T, d)  — one value matrix per group
    Returns  : (H_q, T, d)  — one output per query head
    """
    H_q = Q_heads.shape[0]
    G   = K_groups.shape[0]
    T   = Q_heads.shape[1]
    d   = Q_heads.shape[2]
    queries_per_group = H_q // G

    outputs = np.zeros_like(Q_heads)
    for h in range(H_q):
        # TODO: find the right group index for head h
        g = ???   # replace ??? with the correct expression

        scores  = Q_heads[h] @ K_groups[g].T / math.sqrt(d)   # (T, T)
        mask    = np.triu(np.full((T, T), -1e9), 1)
        scores += mask
        weights = np.exp(scores - scores.max(1, keepdims=True))
        weights /= weights.sum(1, keepdims=True)
        outputs[h] = weights @ V_groups[g]

    return outputs

# Test:
# rng5 = np.random.default_rng(5)
# T_, d_ = 6, 16
# H_q, G = 8, 2
# Q_h = rng5.standard_normal((H_q, T_, d_))
# K_g = rng5.standard_normal((G,   T_, d_))
# V_g = rng5.standard_normal((G,   T_, d_))
# out = gqa_forward(Q_h, K_g, V_g)
# print('Output shape:', out.shape)   # should be (8, 6, 16)

### Exercise 3 — Conceptual questions

Answer in the markdown cell below (or in code comments):

1. **Why does KV Cache only save K and V, not Q?**  
   *(Hint: during autoregressive generation, which token generates the query?)*

2. **MLA caches the compressed latent $\mathbf{c}$ instead of $\mathbf{k},\mathbf{v}$.  
   What is the memory saving ratio if $d_c = 512$ and $H_{kv}=8$, $d_{head}=128$?**

3. **Explain why Streaming LLM keeps "sink" tokens (the very first tokens).  
   What happens if you drop them?** *(See the PPL plots in the lecture.)*

4. **You are building an AI coding assistant that prepends a 2000-token system prompt to every API call.  
   How should you structure your prompts to maximize cache hit rate?  
   What if the system prompt changes daily?**

### Exercise 4 — Pruning policy

Implement a simple H2O-style pruning: after generating each token, keep only the top-K tokens by **cumulative attention score** in the cache.

In [ ]:
def h2o_generate(X, Wq, Wk, Wv, keep_k):
    """
    Autoregressive generation with H2O-style KV cache pruning.
    X      : (T, D) token embeddings
    keep_k : maximum number of KV entries to keep
    Returns: outputs (T, D)
    """
    T, D = X.shape
    outputs   = []
    K_cache   = []   # list of (D,) key vectors
    V_cache   = []   # list of (D,) value vectors
    scores_acc = []  # cumulative attention score per cached token

    for t in range(T):
        k_t = X[t] @ Wk.T
        v_t = X[t] @ Wv.T
        q_t = X[t] @ Wq.T

        K_cache.append(k_t)
        V_cache.append(v_t)
        scores_acc.append(0.0)

        K = np.stack(K_cache)
        V = np.stack(V_cache)
        att = softmax(K @ q_t / math.sqrt(D))   # attention weights

        out_t = att @ V
        outputs.append(out_t)

        # Accumulate attention scores
        for i, a in enumerate(att):
            scores_acc[i] += a

        # TODO: if cache exceeds keep_k, evict the token with the lowest score
        # Hint: use np.argmin(scores_acc) to find the weakest entry
        if len(K_cache) > keep_k:
            pass   # replace this with eviction logic

    return np.stack(outputs)

# Test:
rng6 = np.random.default_rng(11)
T6, D6 = 20, 16
X6  = rng6.standard_normal((T6, D6))
Wq6 = rng6.standard_normal((D6, D6)) * 0.1
Wk6 = rng6.standard_normal((D6, D6)) * 0.1
Wv6 = rng6.standard_normal((D6, D6)) * 0.1

out_full   = kvcache_generate(X6)[0]   # reference (no pruning)
out_pruned = h2o_generate(X6, Wq6, Wk6, Wv6, keep_k=8)

print("Output shapes match:", out_full.shape == out_pruned.shape)
# After implementing pruning, compare outputs — some difference is expected!